# 3A — Baseline: Linear Regression

**Project:** Predictive Modeling of US Used Vehicle Prices  
**Course:** ENGR422 — Applied Machine Learning  
**Author:** *(assign one team member)*  

---

This notebook covers the **Linear Regression baseline** from **Work Package 3 — Model Implementation**.

**Deliverable:** D3.1 — Trained Baseline Linear Regression Model

## 3A.1 — Imports & Load Preprocessed Data

Import the preprocessing pipeline built in Notebook 02. Load the dataset and reproduce the exact same train-test split (same `random_state`).

In [1]:
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path("../data")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42

# Load TRAIN ONLY. The test set is consumed exclusively by notebook 04
# (evaluation) to keep the held-out split untouched during model fitting.
X_train = pd.read_csv(DATA_DIR / "X_train.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze("columns")

print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"Features ({X_train.shape[1]}): {list(X_train.columns)[:8]} ...")


X_train: (305141, 32)
X_test:  (76286, 32)
y_train: (305141,), y_test: (76286,)
Features (32): ['year', 'manufacturer', 'model', 'condition', 'odometer', 'paint_color', 'state', 'fuel_electric'] ...


## 3A.2 — Model Setup

Attach `LinearRegression` from Scikit-Learn as the estimator in the pipeline. This serves as the project baseline — a simple, interpretable model to benchmark against the ensemble methods.

In [ ]:
# Three baseline variants:
#   ols   — plain OLS (no regularization)
#   ridge — L2 regularization, alpha picked by internal 5-fold CV
#   lasso — L1 regularization (also performs implicit feature selection)
# StandardScaler is included so the regularized variants are not skewed by
# differences in feature scale (year vs odometer vs one-hot dummies).

ALPHAS = np.logspace(-3, 3, 13)  # 0.001 ... 1000

models = {
    "ols": Pipeline([
        ("scaler", StandardScaler()),
        ("est", LinearRegression()),
    ]),
    "ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("est", RidgeCV(alphas=ALPHAS, cv=5)),
    ]),
    "lasso": Pipeline([
        ("scaler", StandardScaler()),
        ("est", LassoCV(
            alphas=ALPHAS,
            cv=5,
            random_state=RANDOM_STATE,
            max_iter=10000,
            n_jobs=-1,
        )),
    ]),
}

for name, m in models.items():
    print(f"{name}: {m.steps[-1][1].__class__.__name__}")


## 3A.3 — Training

Fit the full pipeline (preprocessing + Linear Regression) on the training data. Report the training time.

In [ ]:
fit_times = {}
for name, model in models.items():
    print(f"Fitting {name} ...")
    t0 = time.perf_counter()
    model.fit(X_train, y_train)
    fit_times[name] = time.perf_counter() - t0
    print(f"  done in {fit_times[name]:.2f}s")

print(f"\nRidge selected alpha: {models['ridge'].named_steps['est'].alpha_:.4g}")
print(f"Lasso selected alpha: {models['lasso'].named_steps['est'].alpha_:.4g}")


## 3A.4 — Cross-Validation

Run k-fold cross-validation (k=5) on the training set to get a robust estimate of performance and check for high variance across folds. Report MAE and R² per fold.

## 3A.5 — Test Set Evaluation

Predict on the held-out test set and compute all four project metrics:
- **MAE** (Mean Absolute Error)
- **RMSE** (Root Mean Squared Error)
- **R²** (R-squared)
- **MAPE** (Mean Absolute Percentage Error)

Plot predicted vs. actual prices scatter plot and the residuals distribution.

## 3A.6 — Feature Importance / Coefficients

Extract and visualize the learned regression coefficients. Identify the top features driving price predictions in the linear model.

## 3A.7 — Save Model

Serialize the trained pipeline to `models/linear_regression.pkl` using `joblib` for later comparison in the evaluation notebook.

## 3A.8 — Baseline Summary

Briefly summarize:
- The baseline metrics achieved.
- Any obvious limitations (e.g., cannot capture non-linear depreciation).
- What improvement we expect from the tree-based models.